# Treino do detector de nadadores (YOLO11m)

É o notebook que treinou o detector do trabalho, só com as partes de treino: conjunto de dados, treino, avaliação em imagens e treinos extras. As seções de vídeo foram tiradas, porque o sistema final (pasta `sistema_final`) as substituiu. As células de código que ficaram são as que rodaram, sem nenhuma mudança; por isso alguns comentários da configuração ainda falam de vídeos. No fim estão as duas células que foram coladas depois, no computador de treino: mais arquiteturas por competição e o treino final, com as 14 competições, que gerou os pesos do sistema.

## 1. Configuração

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# ---------- onde estão os arquivos ----------
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/TCC-nadador')
    TRABALHO = Path('/content/trabalho')
else:
    BASE = Path(os.environ.get('TCC_BASE', Path.home() / 'TCC-nadador'))
    TRABALHO = BASE / 'trabalho'

ZIP_PATH = BASE / 'Dataset' / 'dataset nadador Yolo11.zip'
VIDEO_DIR = BASE / 'videos'
OUTPUT_DIR = BASE / 'resultados'

# ---------- treino principal ----------
MODEL_NAME = 'yolo11m.pt'
RUN_NAME = 'yolo11m_640_por_competicao'   # troque junto com MODEL_NAME para treinar outra variante
IMGSZ = 640
EPOCHS = 150
PATIENCE = 30
BATCH = -1                     # -1 = maior lote que cabe na GPU; se faltar memória, use 8
WORKERS = 4 if IN_COLAB else 2
FRACTION = 1.0
FORCE_REBUILD = False
FORCE_RETRAIN = False
MERGE_RELAX_INTO_SWIMMER = False

# ---------- treinos extras, feitos DEPOIS de todo o resto (lista vazia = nenhum) ----------
# cada item: (pesos iniciais, nome do treino, resolução)
TREINOS_EXTRAS = [('yolo11s.pt', 'yolo11s_640_por_competicao', 640)]

# ---------- modelos anteriores, para comparar (ignorados se o arquivo não existir) ----------
MODELOS_ANTERIORES = {
    'yolo11x_piscina (11/11)': OUTPUT_DIR / 'yolo11x_swimmer_zip' / 'weights' / 'best.pt',
    'yolo11x_drone (10/11)': OUTPUT_DIR / 'treino_yolo11x' / 'weights' / 'best.pt',
}
MODELO_PARA_VIDEOS = None      # None = usa o modelo treinado neste notebook nas seções 7 a 9

# ---------- vídeos ----------
PASSO_VIDEO = 10
NADADORES = {                  # nadadores visíveis em cada vídeo (None = não contado)
    'natacao_teste_07.mp4': 1, 'natacao_teste_08.mp4': 1, 'natacao_teste_09.mp4': None,
    'natacao_teste_3_c1.mp4': 1, 'natacao_teste_3_c2.mp4': 1,
    'natacao_teste_4_c1.mp4': 8, 'natacao_teste_4_c3.mp4': 8, 'natacao_teste_6_c1.mp4': 8,
}
VIDEOS_RASTREAMENTO = ['natacao_teste_4_c1.mp4', 'natacao_teste_4_c3.mp4', 'natacao_teste_07.mp4',
                       'natacao_teste_08.mp4', 'natacao_teste_3_c1.mp4', 'natacao_teste_3_c2.mp4']
MAX_QUADROS = None             # None = vídeo inteiro

# ---------- rastreador ----------
DET_CONF = 0.10                # corte de confiança do detector
TRACK_ACTIVATION = 0.30
MATCH_THRESHOLD = 0.90         # maior = mais permissivo
LOST_BUFFER = 90
LACUNA_MAX = 60                # quadros: lacunas até esse tamanho são interpoladas dentro da raia

# ---------- contador de braçadas sem pose ----------
LIMIAR_NAO_AGUA = 40           # distância de cor (Lab) para um pixel ser considerado "não é água"
BANDA_BRACADA = (0.4, 2.4)     # Hz: faixa plausível de braçadas por segundo
HISTERESE = 0.0                # fração do desvio-padrão; 0 = cruza a média
RODAR_PIPELINE_DO_REPOSITORIO = True   # seção 9: roda também o contador antigo, por pose, para comparar

# contagem manual, cada braço conta uma; nos vídeos de 8 raias, da raia mais à ESQUERDA para a direita
BRACADAS_MANUAIS = {
    'natacao_teste_07.mp4': [9], 'natacao_teste_08.mp4': [18],
    'natacao_teste_3_c1.mp4': [34], 'natacao_teste_3_c2.mp4': [34],
    'natacao_teste_4_c1.mp4': [9, 6, 8, 5, 7, 8, 6, 5],
    'natacao_teste_4_c3.mp4': [15, 13, 14, 13, 13, 17, 11, 10],
}

# ---------- modo de ensaio: roda tudo em minutos, só para testar o notebook ----------
SMOKE = os.environ.get('TCC_SMOKE') == '1'
if SMOKE:
    MODEL_NAME, RUN_NAME, EPOCHS, FRACTION, BATCH, WORKERS = 'yolo11n.pt', 'ensaio_smoke_v2', 1, 0.03, 4, 0
    TREINOS_EXTRAS = [('yolo11n.pt', 'ensaio_smoke_v2_extra', 640)]
    PASSO_VIDEO, MAX_QUADROS, VIDEOS_RASTREAMENTO = 200, 150, ['natacao_teste_4_c1.mp4', 'natacao_teste_07.mp4']
    MODELO_PARA_VIDEOS = MODELOS_ANTERIORES['yolo11x_piscina (11/11)']

PREPARED_DIR = TRABALHO / 'swimmers_por_competicao'
WORK_DIR = TRABALHO / 'work_raw'
DATA_YAML = PREPARED_DIR / 'data.yaml'
REPO_DIR = Path(os.environ.get('TCC_REPO', TRABALHO / 'tcc-swimmer-tracking'))
RUN_DIR = OUTPUT_DIR / RUN_NAME
print('Colab:', IN_COLAB, '| BASE:', BASE, '| treino:', RUN_DIR)

## 2. Ambiente

In [ ]:
import subprocess

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics',
                    'supervision>=0.30.0,<0.31.0', 'motmetrics'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

import copy
import csv
import io
import json
import re
import shutil
import time
import urllib.request
import warnings
import zipfile
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import supervision as sv
import torch
import ultralytics
import yaml
from IPython.display import display
from scipy.signal import butter, filtfilt
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('ultralytics', ultralytics.__version__, '| supervision', sv.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NÃO HÁ — o treino em CPU é inviável')

assert ZIP_PATH.exists(), f'Não achei o conjunto de dados em {ZIP_PATH}'
assert VIDEO_DIR.exists(), f'Não achei a pasta de vídeos em {VIDEO_DIR}'
TRABALHO.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS = sorted(p for p in VIDEO_DIR.glob('*.mp4'))
VIDEOS_RASTREAMENTO = [v for v in VIDEOS_RASTREAMENTO if (VIDEO_DIR / v).exists()]
print(len(VIDEOS), 'vídeos:', [v.name for v in VIDEOS])

## 3. Conjunto de dados, dividido por competição

Teste: `Angers19`, `Gwangju`, `Nice`. Validação: `StRaphael18`, `7Collines19`. Treino: as outras nove competições.
A célula confere sozinha o resultado esperado: 2.264 / 81 / 58 imagens.

In [ ]:
# Competições reservadas: nenhum quadro delas entra no treino.
TEST_COMPS = {'Angers19', 'Gwangju', 'Nice'}
VAL_COMPS = {'StRaphael18', '7Collines19'}


def origem(p):
    """Quadro de origem: nome do arquivo sem o hash do Roboflow."""
    return re.sub(r'\.rf\.[a-zA-Z0-9]+$', '', p.stem)


def competicao(p):
    base = re.sub(r'_(jpg|jpeg|png)$', '', origem(p))
    prova = re.sub(r'[-_]\d+$', '', base)
    return prova.split('_')[0]


if DATA_YAML.exists() and not FORCE_REBUILD:
    print("Dataset já está preparado. Pulando etapa de reconstrução.")
else:
    print("Iniciando preparação do dataset...")
    if PREPARED_DIR.exists():
        shutil.rmtree(PREPARED_DIR)
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True)

    print("Extraindo ZIP...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(WORK_DIR)

    yaml_files = list(WORK_DIR.rglob('data.yaml'))
    if not yaml_files:
        raise FileNotFoundError("data.yaml não encontrado no ZIP.")
    dataset_root = yaml_files[0].parent
    with open(yaml_files[0], 'r') as f:
        orig_names = yaml.safe_load(f).get('names', [])
    print(f"Classes originais encontradas: {orig_names}")

    class_map = {}
    for i, name in enumerate(orig_names):
        lower_name = name.lower()
        if lower_name in ['bodysurface', 'bodyunder']:
            class_map[i] = 0
        elif lower_name == 'relax':
            class_map[i] = 0 if MERGE_RELAX_INTO_SWIMMER else -1
        elif len(orig_names) == 1:
            class_map[i] = 0
        else:
            print(f"Aviso: classe desconhecida '{name}'. Será ignorada.")
            class_map[i] = -1

    PREPARED_DIR.mkdir(parents=True)
    for s in ['train', 'valid', 'test']:
        (PREPARED_DIR / s / 'images').mkdir(parents=True, exist_ok=True)
        (PREPARED_DIR / s / 'labels').mkdir(parents=True, exist_ok=True)

    ja_usado = set()
    contagem = defaultdict(int)
    caixas = defaultdict(int)
    comps = defaultdict(set)
    for split_orig in ['train', 'valid', 'test']:
        for img in sorted((dataset_root / split_orig / 'images').glob('*.*')):
            comp = competicao(img)
            destino = 'test' if comp in TEST_COMPS else 'valid' if comp in VAL_COMPS else 'train'
            # O treino original tem 9 cópias aumentadas de cada quadro:
            # na validação e no teste entra só uma cópia por quadro.
            if destino != 'train' and split_orig == 'train':
                if origem(img) in ja_usado:
                    continue
                ja_usado.add(origem(img))
            shutil.copy(img, PREPARED_DIR / destino / 'images' / img.name)
            lbl_src = dataset_root / split_orig / 'labels' / f"{img.stem}.txt"
            lbl_dst = PREPARED_DIR / destino / 'labels' / f"{img.stem}.txt"
            linhas = lbl_src.read_text().splitlines() if lbl_src.exists() else []
            with open(lbl_dst, 'w') as f:
                for line in linhas:
                    parts = line.split()
                    if not parts:
                        continue
                    novo = class_map.get(int(parts[0]), -1)
                    if novo != -1:
                        f.write(f"{novo} {' '.join(parts[1:])}\n")
                        caixas[destino] += 1
            contagem[destino] += 1
            comps[destino].add(comp)

    new_yaml = {
        'path': str(PREPARED_DIR.absolute()),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 1,
        'names': ['swimmer'],
    }
    with open(DATA_YAML, 'w') as f:
        yaml.dump(new_yaml, f, sort_keys=False)

    shutil.rmtree(WORK_DIR)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print("Imagens por divisão:", dict(contagem))
    print("Caixas de nadador por divisão:", dict(caixas))
    for s in ['train', 'valid', 'test']:
        print(f"  {s}: {sorted(comps[s])}")
    esperado = {'train': 2264, 'valid': 81, 'test': 58}
    assert dict(contagem) == esperado, f"contagem diferente da esperada {esperado}"
    assert not (comps['train'] & comps['valid']) and not (comps['train'] & comps['test'])
    print("✅ Dataset preparado, dividido por competição.")

## 4. Treino

Não treina de novo se já existe treino concluído com o mesmo nome; retoma se o treino parou no meio.

In [ ]:
def treina(model_name, run_name, imgsz):
    """Treina (ou retoma, ou reaproveita) e devolve o caminho do best.pt."""
    run_dir = OUTPUT_DIR / run_name
    last, best = run_dir / 'weights' / 'last.pt', run_dir / 'weights' / 'best.pt'
    estado = 'NOVO'
    if last.exists():
        ckpt = torch.load(last, map_location='cpu', weights_only=False)
        estado = 'CONCLUIDO' if (ckpt.get('epoch', 0) == -1 or not ckpt.get('optimizer')) else 'RETOMAR'
    if estado == 'CONCLUIDO' and FORCE_RETRAIN:
        estado = 'NOVO'
    print(f'{run_name}: {estado}')
    if estado == 'RETOMAR':
        YOLO(str(last)).train(resume=True)
    elif estado == 'NOVO':
        YOLO(model_name).train(
            data=str(DATA_YAML), epochs=EPOCHS, imgsz=imgsz, batch=BATCH, device=DEVICE, workers=WORKERS,
            project=str(OUTPUT_DIR), name=run_name, exist_ok=True, patience=PATIENCE, cos_lr=True,
            close_mosaic=10, cache='disk', amp=True, plots=True, seed=42, fraction=FRACTION,
        )
    assert best.exists(), f'O best.pt de {run_name} não foi gerado.'
    return best


BEST_MODEL = treina(MODEL_NAME, RUN_NAME, IMGSZ)
print('Melhor modelo:', BEST_MODEL)

## 5. Avaliação em imagens anotadas

O modelo deste notebook nunca viu as competições da validação e do teste. O `yolo11x_piscina (11/11)` viu, então o número dele é otimista.

In [ ]:
def avalia(pesos, split, imgsz=IMGSZ):
    r = YOLO(str(pesos)).val(data=str(DATA_YAML), split=split, imgsz=imgsz, device=DEVICE, plots=False,
                             verbose=False, workers=0, project=str(TRABALHO / 'val_tmp'), name='v', exist_ok=True)
    return {'precisao': round(float(r.box.mp), 3), 'recall': round(float(r.box.mr), 3),
            'mAP50': round(float(r.box.map50), 3), 'mAP50_95': round(float(r.box.map), 3)}


MODELOS = {RUN_NAME: BEST_MODEL}
MODELOS.update({nome: p for nome, p in MODELOS_ANTERIORES.items() if Path(p).exists()})
linhas = []
for nome, pesos in MODELOS.items():
    for split in ['val', 'test']:
        linhas.append({'modelo': nome, 'split': split, **avalia(pesos, split)})
tabela_imagens = pd.DataFrame(linhas)
tabela_imagens.to_csv(RUN_DIR / 'avaliacao_imagens.csv', index=False)
display(tabela_imagens)

## 6. Treinos extras

Treina as variantes listadas em `TREINOS_EXTRAS` e mede cada uma na validação e no teste, ao lado do treino principal. Serve para a tabela de comparação entre tamanhos de modelo do TCC.

In [ ]:
linhas = [{'modelo': RUN_NAME, 'imgsz': IMGSZ, 'split': s, **avalia(BEST_MODEL, s)} for s in ['val', 'test']]
for pesos_iniciais, nome_treino, imgsz in TREINOS_EXTRAS:
    melhor = treina(pesos_iniciais, nome_treino, imgsz)
    for s in ['val', 'test']:
        linhas.append({'modelo': nome_treino, 'imgsz': imgsz, 'split': s, **avalia(melhor, s, imgsz)})
tabela_extras = pd.DataFrame(linhas)
tabela_extras.to_csv(RUN_DIR / 'comparacao_treinos.csv', index=False)
display(tabela_extras)

## 7. Onde ficou cada coisa

Tudo em `resultados/<RUN_NAME>/`: `weights/best.pt`, `avaliacao_imagens.csv` e `comparacao_treinos.csv`. O treino final fica em `resultados/yolo11m_640_final_14_competicoes/` (`weights/last.pt`, `weights/modelo_final.pt` e `como_foi_treinado.json`).

In [ ]:
for arq in sorted(RUN_DIR.rglob('*')):
    if arq.is_file() and arq.suffix in {'.csv', '.pt', '.mp4'} and 'cache_deteccoes' not in arq.parts:
        print(f'{arq.stat().st_size / 1e6:8.1f} MB  {arq.relative_to(OUTPUT_DIR)}')

## 8. Mais arquiteturas por competição

Célula colada no fim do notebook, no computador de treino, para os três treinos extras que não tinham chegado a rodar: YOLO11n a 640, YOLO11s a 960 e YOLO11x a 640.

In [ ]:
# ===== Fila de arquiteturas por competição (os três treinos que não chegaram a rodar) =====
# Cole no fim do notebook e execute. Grava a tabela em resultados\comparacao_treinos_extras.csv
# (na pasta resultados, que sempre existe, e não mais dentro da pasta de um treino).
MAIS_TREINOS = [
    ('yolo11n.pt', 'yolo11n_640_por_competicao', 640),
    ('yolo11s.pt', 'yolo11s_960_por_competicao', 960),
    ('yolo11x.pt', 'yolo11x_640_por_competicao', 640),
]
linhas = []
for pesos_iniciais, nome_treino, imgsz in MAIS_TREINOS:
    try:
        melhor = treina(pesos_iniciais, nome_treino, imgsz)
        for s in ['val', 'test']:
            linhas.append({'modelo': nome_treino, 'imgsz': imgsz, 'split': s, **avalia(melhor, s, imgsz)})
    except Exception as erro:
        print(nome_treino, 'falhou:', erro)
    try:
        pd.DataFrame(linhas).to_csv(OUTPUT_DIR / 'comparacao_treinos_extras.csv', index=False)
    except Exception as erro:
        print('não consegui gravar a tabela:', erro)
display(pd.DataFrame(linhas))


## 9. Treino final, com as 14 competições

Célula colada no fim do notebook, no computador de treino. Junta as imagens de todas as competições no treino e treina com número fixo de épocas, sem parada antecipada; o modelo final é o `last.pt` e, por isso, não tem nota de teste (a nota da receita sai da validação cruzada, em `validacao_cruzada_detector.py`).

No treino que gerou os pesos do sistema, a escolha automática não rodou, porque as pastas dos treinos por competição tinham mudado de lugar. As arquiteturas e as épocas foram informadas à mão em `FINAL_FORCAR`, com os mesmos valores que a escolha automática dá: YOLO11s a 640 com 54 épocas e YOLO11m a 640 com 25. O sistema usa o YOLO11m. Para repetir só esse treino: `FINAL_FORCAR = [('yolo11m.pt', 640, 25)]`.

In [ ]:
# ===== TREINO FINAL: mesma receita, agora com as 14 competições =====
# Cole esta célula no FIM do notebook que está treinando e mande rodar: ela entra na fila
# e só começa quando os treinos por competição terminarem.
#
# - As arquiteturas são escolhidas pela VALIDAÇÃO dos treinos por competição (nunca pelo teste),
#   com a mesma nota que o Ultralytics usa para escolher o best.pt (mAP50-95).
# - O modelo final NÃO tem nota de teste: todas as imagens entram no treino. A nota que vai
#   para o TCC continua sendo a do treino por competição da mesma arquitetura.
# - O número de épocas é fixo, calculado a partir da melhor época do treino por competição.
#   Não há parada antecipada e o modelo final é o last.pt.
import json
import math

FINAL_FORCAR = None            # None = escolhe sozinho; ou uma lista como [('yolo11s.pt', 640, 54)] = (pesos, resolução, épocas)
N_FINAIS = 2                   # quantas arquiteturas treinar (as melhores na validação); a decisão entre elas sai dos vídeos
EXT_IMAGEM = {'.jpg', '.jpeg', '.png'}   # a pasta de imagens também guarda o cache .npy do Ultralytics
COPIAS = 9                     # no ZIP cada quadro do treino tem 9 cópias; os da validação e do teste têm 1
FINAL_DIR = TRABALHO / 'swimmers_todas'
FINAL_YAML = FINAL_DIR / 'data.yaml'
FINAL_RAW = TRABALHO / 'work_raw_final'


def imagens(pasta):
    return sorted(p for p in Path(pasta).glob('*.*') if p.suffix.lower() in EXT_IMAGEM)


def candidatos_por_competicao():
    """Treinos por competição concluídos, com a melhor época pela nota de validação (mAP50-95)."""
    linhas = []
    for run in sorted(OUTPUT_DIR.glob('*_por_competicao')):
        res, args, last = run / 'results.csv', run / 'args.yaml', run / 'weights' / 'last.pt'
        if not (res.exists() and args.exists() and last.exists()):
            continue
        try:
            ckpt = torch.load(last, map_location='cpu', weights_only=False)
        except Exception as erro:
            print(f'{run.name}: não consegui ler o last.pt ({erro}); ignorado')
            continue
        if not (ckpt.get('epoch', 0) == -1 or not ckpt.get('optimizer')):
            print(f'{run.name}: treino incompleto; ignorado')
            continue
        r = pd.read_csv(res)
        r.columns = [c.strip() for c in r.columns]
        nota = r['metrics/mAP50-95(B)']
        i = int(nota.idxmax())
        a = yaml.safe_load(args.read_text(encoding='utf-8'))
        linhas.append({'treino': run.name, 'pesos_iniciais': Path(str(a['model'])).name, 'imgsz': int(a['imgsz']),
                       'melhor_epoca': int(r['epoch'][i]), 'epocas_rodadas': len(r),
                       'val_mAP50': round(float(r['metrics/mAP50(B)'][i]), 3),
                       'val_mAP50_95': round(float(r['metrics/mAP50-95(B)'][i]), 3), 'nota_val': round(float(nota[i]), 4)})
    return pd.DataFrame(linhas)


def prepara_todas():
    """Todas as imagens do ZIP no treino, classe única; quadros que só têm 1 cópia são repetidos COPIAS vezes."""
    if FINAL_YAML.exists() and not FORCE_REBUILD:
        print('Conjunto com as 14 competições já preparado.')
        return
    for pasta in (FINAL_DIR, FINAL_RAW):
        if pasta.exists():
            shutil.rmtree(pasta)
    FINAL_RAW.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(FINAL_RAW)
    yaml_orig = list(FINAL_RAW.rglob('data.yaml'))[0]
    raiz = yaml_orig.parent
    nomes = yaml.safe_load(yaml_orig.read_text()).get('names', [])
    mapa = {}
    for i, nome in enumerate(nomes):
        n = nome.lower()
        mapa[i] = 0 if n in ('bodysurface', 'bodyunder') or len(nomes) == 1 else (0 if n == 'relax' and MERGE_RELAX_INTO_SWIMMER else -1)
    for s in ('train', 'valid'):
        (FINAL_DIR / s / 'images').mkdir(parents=True)
        (FINAL_DIR / s / 'labels').mkdir(parents=True)

    def grava(img, lbl_src, destino, nome):
        shutil.copy(img, FINAL_DIR / destino / 'images' / f'{nome}{img.suffix}')
        linhas = lbl_src.read_text().splitlines() if lbl_src.exists() else []
        with open(FINAL_DIR / destino / 'labels' / f'{nome}.txt', 'w') as f:
            for linha in linhas:
                p = linha.split()
                if p and mapa.get(int(p[0]), -1) != -1:
                    f.write(f"0 {' '.join(p[1:])}\n")

    copias_por_quadro = defaultdict(int)
    for split in ('train', 'valid', 'test'):
        for img in imagens(raiz / split / 'images'):
            copias_por_quadro[origem(img)] += 1
    n_imagens, quadros, comps = 0, set(), set()
    for split in ('train', 'valid', 'test'):
        for img in imagens(raiz / split / 'images'):
            lbl = raiz / split / 'labels' / f'{img.stem}.txt'
            repeticoes = COPIAS if copias_por_quadro[origem(img)] == 1 else 1
            for k in range(repeticoes):
                grava(img, lbl, 'train', img.stem if repeticoes == 1 else f'{img.stem}_copia{k + 1}')
                n_imagens += 1
            quadros.add(origem(img))
            comps.add(competicao(img))
            if split != 'train':           # o Ultralytics exige uma validação; esta está DENTRO do treino
                grava(img, lbl, 'valid', img.stem)
    with open(FINAL_YAML, 'w') as f:
        yaml.dump({'path': str(FINAL_DIR.absolute()), 'train': 'train/images', 'val': 'valid/images',
                   'nc': 1, 'names': ['swimmer']}, f, sort_keys=False)
    shutil.rmtree(FINAL_RAW)
    print(f'Conjunto final: {n_imagens} imagens de {len(quadros)} quadros distintos, {len(comps)} competições: {sorted(comps)}')
    assert len(quadros) == 467 and len(comps) == 14, 'esperava 467 quadros distintos de 14 competições'


tabela_candidatos = candidatos_por_competicao()
if len(tabela_candidatos):
    tabela_candidatos = tabela_candidatos.sort_values('nota_val', ascending=False).reset_index(drop=True)
    display(tabela_candidatos)
prepara_todas()
N_FINAL = len(imagens(FINAL_DIR / 'train' / 'images'))
N_POR_COMPETICAO = len(imagens(PREPARED_DIR / 'train' / 'images'))

if FINAL_FORCAR:
    FINAIS = [(p, int(tam), int(ep), 'definido à mão em FINAL_FORCAR') for p, tam, ep in FINAL_FORCAR]
else:
    assert len(tabela_candidatos), 'Nenhum treino por competição concluído: preencha FINAL_FORCAR.'
    # mesmo número de passos de treino da melhor época do treino por competição, mais 20%
    FINAIS = [(c['pesos_iniciais'], int(c['imgsz']), max(20, math.ceil(1.2 * c['melhor_epoca'] * N_POR_COMPETICAO / N_FINAL)),
               f"{c['treino']} (melhor época {c['melhor_epoca']}, mAP50-95 de validação {c['val_mAP50_95']})")
              for c in tabela_candidatos.head(N_FINAIS).to_dict('records')]

MODELOS_FINAIS = {}
for PESOS_FINAL, IMGSZ_FINAL, EPOCAS_FINAL, MOTIVO in FINAIS:
    NOME_FINAL = f"{Path(PESOS_FINAL).stem}_{IMGSZ_FINAL}_final_14_competicoes"
    print(f'\nTreino final: {PESOS_FINAL} a {IMGSZ_FINAL}, {EPOCAS_FINAL} épocas, {N_FINAL} imagens -> {NOME_FINAL}')
    print('Escolhido a partir de:', MOTIVO)
    try:
        run_final = OUTPUT_DIR / NOME_FINAL
        last_final = run_final / 'weights' / 'last.pt'
        estado = 'NOVO'
        if last_final.exists():
            ckpt = torch.load(last_final, map_location='cpu', weights_only=False)
            estado = 'CONCLUIDO' if (ckpt.get('epoch', 0) == -1 or not ckpt.get('optimizer')) else 'RETOMAR'
        print(f'{NOME_FINAL}: {estado}')
        if estado == 'RETOMAR':
            YOLO(str(last_final)).train(resume=True)
        elif estado == 'NOVO':
            YOLO(PESOS_FINAL).train(
                data=str(FINAL_YAML), epochs=EPOCAS_FINAL, imgsz=IMGSZ_FINAL, batch=BATCH, device=DEVICE, workers=WORKERS,
                project=str(OUTPUT_DIR), name=NOME_FINAL, exist_ok=True, patience=0, cos_lr=True,
                close_mosaic=10, cache='disk', amp=True, plots=True, seed=42, fraction=FRACTION,
            )
        assert last_final.exists(), 'O last.pt do treino final não foi gerado.'
        modelo_final = run_final / 'weights' / 'modelo_final.pt'
        shutil.copy(last_final, modelo_final)
        (run_final / 'como_foi_treinado.json').write_text(json.dumps({
            'modelo_final': str(modelo_final), 'pesos_iniciais': PESOS_FINAL, 'imgsz': IMGSZ_FINAL, 'epocas': EPOCAS_FINAL,
            'imagens_no_treino': N_FINAL, 'quadros_distintos': 467, 'competicoes': 14, 'escolhido_a_partir_de': MOTIVO,
            'aviso': 'Sem nota de teste: todas as imagens entraram no treino. A validação exibida durante o treino está dentro do treino.',
        }, ensure_ascii=False, indent=2), encoding='utf-8')
        MODELOS_FINAIS[NOME_FINAL] = modelo_final
        print('✅ Modelo final:', modelo_final)
    except Exception as erro:
        print(f'{NOME_FINAL} falhou: {erro}')
print('\nModelos finais prontos:', {k: str(v) for k, v in MODELOS_FINAIS.items()})